In [1]:
import json
import time
from pathlib import Path
import chromadb
from chromadb.utils import embedding_functions
import ollama

In [2]:
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_collection(
        name="my_docs",
        embedding_function=embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )
    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:

def answer_question(user_query, collection, top_k=3): 
    
    results = collection.query(
        query_texts=[user_query],
        n_results=top_k
    ) 
    chunks = results['documents'][0]
    sources = results['metadatas'][0]
    distances = results['distances'][0]
    
    context_parts = []
    for chunk, meta, dist in zip(chunks, sources, distances):
      
        context_parts.append(
            f"[ДЖЕРЕЛО: {meta['source']}, СТОРІНКА: {meta['page']}, РЕЛЕВАНТНІСТЬ: {1-dist:.2f}]\n{chunk}"
        )
    
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f""" 
            You are an assistant who responds ONLY based on the context provided.
        RULES:
        1. Answer ONLY based on the context below.
        2. If the information is NOT in the context, honestly say “I don't know” or “This information is not included in the provided documents.”
        3. DO NOT make up information that is not in the context.
        4. If you answer, cite your source.

        CONTEXT (taken from documents):
        {context}

        USER QUESTION:
        {user_query}

        ANSWER (based on the context):
    """
         
    # query for llama3.2 model
    try:
        import ollama 
        response = ollama.chat(
            model = 'llama3.2',
            
            messages=[
                {"role": "system", "content": "You are an assistant. Respond only based on the context provided. If you don't know, say 'I don't know.'"},
                {"role": "user", "content": prompt}
            ],
            options = {
                'temperature': 0.3, # To more accuracy answer
                'num_predict': 500  # To more better think, but longer
            }   
        )
        answer = response['message']['content']
    except:
        print(f"Ollama недоступний")
        answer = "[ПОТРІБНА LLM МОДЕЛЬ]"
     
    return {
        'answer': answer,
        'sources': sources,
        'chunks': chunks,
        'context': context,
        'prompt': prompt
    }

In [4]:
def calculate_metrics(question, retrieved_results, generated_answer):
     
    found_sources = [meta['source'] for meta in retrieved_results['metadatas'][0]]
    expected_sources = question.get('expected_sources', [])
    source_found = any(exp in str(found_sources) for exp in expected_sources)
 
    source_found = any(exp in src
                       for exp in expected_sources
                       for src in found_sources)
    
    keywords = question.get('expected_keywords', [])
    if keywords:
        all_text     = " ".join(retrieved_results['documents'][0]).lower()
        found_kw     = [kw for kw in keywords if kw.lower() in all_text]
        grounded     = len(found_kw) / len(keywords) >= 0.5
    else:
        found_kw = []
        grounded = False
          
    return {
        'source_found':      source_found,    
        'grounded':          grounded,          
        'expected_sources':  expected_sources,
        'found_sources':     found_sources,
        'expected_keywords': keywords,
        'found_keywords':    found_kw,
    }


In [5]:
def run_evaluation(questions, collection, answer_function=None):
    all_results = []
    
    for idx, question in enumerate(questions, 1):

        print(f"\n[{idx}/{len(questions)}]: {question['question'][:60]}")
         
        start_time = time.time()
        results = collection.query(
            query_texts=[question['question']],
            n_results=3  # top-k = 3
        ) 
    
        answer_result = answer_function(question['question'], collection)
        generated_answer = answer_result['answer'] 
        
        end_time = time.time()
        latency = end_time - start_time

        metrics = calculate_metrics(question, results, generated_answer)
          
        src_mark = "[+]" if metrics['source_found'] else "[-]"
        kw_mark  = "[+]" if metrics['grounded']     else "[-]"
        
        print(f"answer:   {generated_answer}")
        print(f"  Latency:          {latency:.2f}s")
        print(f"  {src_mark} Source found:  {metrics['source_found']}")
        print(f"      Expected: {metrics['expected_sources']}")
        print(f"      Found:    {metrics['found_sources']}")
        print(f"  {kw_mark} Grounded:      {metrics['grounded']}")
        print(f"      Expected kw: {metrics['expected_keywords']}")
        print(f"      Found kw:    {metrics['found_keywords']}")


        all_results.append({
            'id':       question['id'],
            'question': question['question'],
            'answer':   generated_answer,
            'metrics':  metrics,
            'latency':  round(latency, 3),
        })
         
    return all_results


In [7]:
with open('Evaluation_question.json', 'r', encoding='utf-8') as f:
        questions = json.load(f)
print('length all question: ', len(questions))
 
results = run_evaluation(questions, collection, answer_question)
results

length all question:  20

[1/20]: What is the F1 score of DeepSeek-R1 on 5-class sentiment?
answer:   I don't know. The question about the F1 score of DeepSeek-R1 on 5-class sentiment is not included in the provided documents. However, according to Table 1, the full DeepSeek-R1 model achieved a 91.39% F1 score with 30-shot learning for the five-level sentiment classification task (see Table 1).
  Latency:          16.18s
  [+] Source found:  True
      Expected: ['2503.11655v5.pdf']
      Found:    ['2503.11655v5.pdf', '2503.11655v5.pdf', '2503.11655v5.pdf']
  [+] Grounded:      True
      Expected kw: ['91.39', 'F1', 'sentiment']
      Found kw:    ['91.39', 'F1', 'sentiment']

[2/20]: What is the accuracy of DeepSeek-R1 on binary tasks?
answer:   I don't know. The provided documents do not mention the accuracy of DeepSeek-R1 on binary tasks.
  Latency:          3.47s
  [-] Source found:  False
      Expected: ['2025.unlp-1.18.pdf']
      Found:    ['2503.11655v5.pdf', '2503.11655v5.p

[{'id': 1,
  'question': 'What is the F1 score of DeepSeek-R1 on 5-class sentiment?',
  'answer': "I don't know. The question about the F1 score of DeepSeek-R1 on 5-class sentiment is not included in the provided documents. However, according to Table 1, the full DeepSeek-R1 model achieved a 91.39% F1 score with 30-shot learning for the five-level sentiment classification task (see Table 1).",
  'metrics': {'source_found': True,
   'grounded': True,
   'expected_sources': ['2503.11655v5.pdf'],
   'found_sources': ['2503.11655v5.pdf',
    '2503.11655v5.pdf',
    '2503.11655v5.pdf'],
   'expected_keywords': ['91.39', 'F1', 'sentiment'],
   'found_keywords': ['91.39', 'F1', 'sentiment']},
  'latency': 16.182},
 {'id': 2,
  'question': 'What is the accuracy of DeepSeek-R1 on binary tasks?',
  'answer': "I don't know. The provided documents do not mention the accuracy of DeepSeek-R1 on binary tasks.",
  'metrics': {'source_found': False,
   'grounded': False,
   'expected_sources': ['2025.u